# Color Analysis Testing

- Step 1: Separate single-color products from multi-color products
    - Develop a preview product function to see the actual product and its details (`product_name` and `product_id` from products.csv, the first `variant_image_url` of `product_id` from shades.csv). Use this function to quickly go which product is single-color only.

## Setup

In [ ]:
from IPython.display import display, HTML
import pandas as pd

## Load Data

In [2]:
encodings = ["utf-8-sig", "utf-8", "cp1252", "iso-8859-1"]
## checking for encoding
for enc in encodings:
    try:
        df = pd.read_csv(r"C:\Users\Thao\Documents\Ulta Analysis\data\raw\blush\20260724T001508Z\variants.csv", encoding=enc)
        print("OK with:", enc)
        break
    except UnicodeDecodeError as e:
        print("Failed:", enc)

OK with: utf-8-sig


In [3]:
# products = pd.read_csv(r"C:\Users\Thao\Documents\Ulta Analysis\data\raw\blush\20260724T001508Z\products.csv", encoding="utf-8-sig")
variants = pd.read_csv(r"C:\Users\Thao\Documents\Ulta Analysis\data\raw\blush\20260724T001508Z\variants.csv", encoding="utf-8-sig")

## Preparing the Dataset

In [ ]:
variants_instock = variants[~variants["availability"].astype(str).str.contains("OutOfStock", na=False)] # filter out OutOfStock product_id

# group by product_id, and returns only product_id and variant_image_url in the order of product_id
df = (variants_instock.sort_values(["product_id"])  # optional; keeps deterministic order if your input already is
.groupby("product_id", as_index=False)
.first()[["product_id", "variant_image_url"]])

In [ ]:
# Drop rows where sku_id is 2616562 or 77008681
variants_instock = variants[~variants['sku_id'].isin([2616562, 77008681])] # these sku_id don't have their swatch_image_url

In [ ]:
variants_instock = variants_instock[~variants_instock['product_id'].isin(["mkt77009832", "mkt77009894"])] # these product_id are brushes not blush

In [254]:
# 1) keep only in-stock rows
variants_instock = variants[~variants["availability"].astype(str).str.contains("OutOfStock", na=False)]

# 2) for each product_id, pick the first variant_image_url (deterministic via sort)
df = (variants_instock
      .sort_values("product_id")
      .groupby("product_id", sort=False)["variant_image_url"]
      .first())

## Define Functions

In [258]:
# From list of product_id to dict of product_id and variant_image_url
def list_to_image(list):
    selected_urls = df.reindex(list)
    # print(selected_urls)
    print(f"Number of product_id: {len(selected_urls)}")
    return [
        {"product_id": pid, "variant_image_url": selected_urls.loc[pid] }
        for pid in list
    ]

In [343]:
def show_swatches(dataset, start: int = 0, stop: int = 20, variants_df=None):
    # Convert pandas DataFrame / Series / list-like -> list[dict]
    if hasattr(dataset, "to_dict"):
        # DataFrame: to_dict("records") works
        if hasattr(dataset, "columns"):
            dataset = dataset.to_dict("records")
        else:
            # Series: convert index/value pairs into the expected dict format
            dataset = [
                {"product_id": idx, "variant_image_url": val}
                for idx, val in dataset.items()
            ]

    # Handle case where dataset is a list of strings (product IDs)
    # Convert to list of dicts if needed
    if dataset and isinstance(dataset[0], str):
        dataset = [{"product_id": pid, "variant_image_url": ""} for pid in dataset]

    # If dicts have no availability (e.g., dataset1), skip the filter gracefully
    filtered = []
    for x in dataset:
        if "availability" in x:
            if "OutOfStock" in str(x.get("availability", "")):
                continue
        filtered.append(x)

    # Get swatch images from variants DataFrame if provided
    product_swatches = {}
    if variants_df is not None:
        # Get unique product IDs from the filtered dataset
        for item in filtered:
            pid = str(item.get("product_id", ""))
            
            # Find all rows matching this product_id
            mask = variants_df["product_id"].astype(str) == pid
            matching_rows = variants_df[mask]
            
            # Collect valid swatch URLs
            swatches = []
            for _, row in matching_rows.iterrows():
                swatch_url = row.get("swatch_image_url", "")
                # Check if swatch_url is valid (not NaN, not empty)
                if pd.notna(swatch_url) and str(swatch_url).strip():
                    swatches.append(str(swatch_url))
            
            # Store unique swatches (remove duplicates)
            if swatches:
                product_swatches[pid] = list(dict.fromkeys(swatches))

    html = "<div style='display: grid; grid-template-columns: repeat(5, 1fr); gap: 10px;'>"
    for item in filtered[start:stop]:
        pid = str(item.get("product_id", ""))
        url = item.get("variant_image_url", "")
        
        # If no variant_image_url, try to get one from variants_df
        if not url and variants_df is not None:
            mask = variants_df["product_id"].astype(str) == pid
            matching_rows = variants_df[mask]
            if not matching_rows.empty:
                first_url = matching_rows.iloc[0].get("variant_image_url", "")
                if pd.notna(first_url):
                    url = first_url
        
        swatches = product_swatches.get(pid, [])

        html += f"""
        <div style='text-align: center;'>
            <img src="{url}" style='width: 200px; height: 200px; object-fit: cover; border-radius: 8px;'>
            <p style='font-size: 12px; margin-top: 5px;'>{pid}</p>
        """
        
        # Add swatch images if available
        if swatches:
            html += "<div style='display: flex; justify-content: center; gap: 5px; margin-top: 5px; flex-wrap: wrap;'>"
            for swatch_url in swatches:
                html += f"""
                    <img src="{swatch_url}" 
                         style='width: 40px; height: 40px; object-fit: cover; border-radius: 4px; border: 1px solid #ddd;'
                         title="Swatch">
                """
            html += "</div>"
        
        html += "</div>"

    html += "</div>"
    display(HTML(html))

In [ ]:
def remove_ids_from_list(list1, check_dict = None, dict_key = None,):
    
    to_remove = set(check_dict.get(dict_key, []))
    return [pid for pid in list1 if pid not in to_remove]


In [ ]:
def remove_ids_from_list_v2(list1, *args, check_dict=None, dict_key=None):
    """
    Remove IDs from list1 that are found in any of the provided sources.
    
    Args:
        list1: List of IDs to filter
        *args: Additional lists to use as removal sources
        check_dict: Dictionary containing lists of IDs (optional)
        dict_key: Key to use if check_dict is provided (optional)
    
    Returns:
        Filtered list with IDs removed
    
    Examples:
        # Multiple lists
        remove_ids_from_list(my_list, list2, list3, list4)
        
        # Dict + additional lists
        remove_ids_from_list(my_list, list2, check_dict=palettes, dict_key="with_swatch_image_url")
        
        # Just a dict
        remove_ids_from_list(my_list, check_dict=palettes, dict_key="no_swatch_image_url")
    """
    # Build the removal set
    removal_set = set()
    
    # Add all IDs from additional lists
    for additional_list in args:
        if additional_list is not None:
            if isinstance(additional_list, (list, set, tuple)):
                removal_set.update(additional_list)
            else:
                removal_set.add(additional_list)
    
    # Add IDs from dictionary if provided
    if check_dict is not None and dict_key is not None:
        removal_set.update(check_dict.get(dict_key, []))
    
    # Filter and return
    return [pid for pid in list1 if pid not in removal_set]

In [405]:
def remove_ids_from_list_v3(list_to_filter=None, *args, to_remove=None, check_dict=None, dict_key=None):
    """
    Remove IDs from a target list that are found in any of the provided sources.
    
    Args:
        list_to_filter: The list to filter (first positional argument)
        *args: Additional lists to use as removal sources
        to_remove: Direct list/set of IDs to remove (keyword argument)
        check_dict: Dictionary containing lists of IDs (optional)
        dict_key: Key to use if check_dict is provided (optional)
    
    Returns:
        Filtered list with IDs removed
    
    Examples:
        # Remove list2 elements from list3
        remove_ids_from_list(list3, list2)
        
        # Remove from list1 using multiple sources
        remove_ids_from_list(list1, list2, list3)
        
        # Using keyword arguments
        remove_ids_from_list(my_list, to_remove=blacklist)
        
        # Using dict
        remove_ids_from_list(my_list, check_dict=palettes, dict_key="with_swatch_image_url")
    """
    if list_to_filter is None:
        raise ValueError("No target list provided to filter")
    
    # Build the removal set
    removal_set = set()
    
    # Add all IDs from additional lists (positional args after the first one)
    for additional_list in args:
        if additional_list is not None:
            if isinstance(additional_list, (list, set, tuple)):
                removal_set.update(additional_list)
            else:
                removal_set.add(additional_list)
    
    # Add IDs from direct to_remove parameter
    if to_remove is not None:
        if isinstance(to_remove, (list, set, tuple)):
            removal_set.update(to_remove)
        else:
            removal_set.add(to_remove)
    
    # Add IDs from dictionary if provided
    if check_dict is not None and dict_key is not None:
        removal_set.update(check_dict.get(dict_key, []))
    
    # Filter and return
    return [pid for pid in list_to_filter if pid not in removal_set]

In [344]:
def check_swatch_image_url(list1, df=variants):
    """
    Separates product IDs into those with and without swatch image URLs.
    
    Args:
        list1: List of product IDs to check
        df: DataFrame containing product data (default: variants)
    
    Returns:
        tuple: (no_swatch_url_list, with_swatch_url_list)
    """
    # Filter to only relevant products
    subset = df[df["product_id"].astype(str).isin(list1)]
    
    # Products without swatch image (NaN or blank/whitespace)
    null_mask = (
        subset["swatch_image_url"].isna() | 
        (subset["swatch_image_url"].astype(str).str.strip() == "")
    )
    no_swatch_url = subset[null_mask]["product_id"].astype(str).unique().tolist()
    
    # Products with valid swatch image
    with_swatch_url = subset[~null_mask]["product_id"].astype(str).unique().tolist()
    
    return no_swatch_url, with_swatch_url

In [105]:
## mult_colored_blush, set_blush, mult_colors_tools, trio_combo, blush_combo, small_swatch_image
## Remaining products that are been categorized into one of the lists above (aka, these products only has single-color)
def single_color_product():
    all_ids = set().union(
        # set(mult_colored_blush),
        set(set_blush),
        set(mult_colors_tools),
        set(trio_combo),
        set(blush_combo),
        set(small_swatch_image)
    )

    df_filtered = df[~df.index.astype(str).isin(all_ids)]
    remaining_product_ids = df_filtered.index.astype(str).tolist()
    print(f"Number of Single-color Products: {len(remaining_product_ids)}")
    return remaining_product_ids

remaining_product_ids = single_color_product()

Number of Single-color Products: 246


## Categorize product_id

Putting `product_id`s into different groups:
- For color analysis, putting `product_id` with single colors into 1 group. The others in different groupss

### Stage 1

In [ ]:
### Stage 1
## swatch_image_url has multiple colors instead of a singular color
mult_colored_blush = [
    "mkt77000411", "mkt77001487", "mkt77001819", "mkt77001820", "mkt77002358", "mkt77004845"
    , "mkt77007311", "pimprod2020448", "pimprod2037175", "pimprod2042910", "pimprod2043903"
    , "pimprod2049565", "pimprod2050040", "pimprod2050058", "pimprod2050643", "pimprod2050867"
    , "pimprod2051122", "pimprod2052290", "pimprod2053334", "pimprod2053436", "pimprod2055456"
    , "pimprod2056002", "pimprod2056649", "pimprod2056650", "pimprod2058301", "pimprod2059581"
    , "pimprod2059602", "xlsImpprod14521197", "pimprod2046620", "mkt77007645", "mkt77004019"
    , "mkt77004845"
]


# # no swatch_image_url or the swatch_image_url don't have the colors
set_blush = [
    "mkt77002598", "mkt77004019", "mkt77008621"
    , "mkt77008994", "pimprod2018302", "pimprod2037933", "pimprod2053121"
    , "pimprod2057726", "pimprod2058294", "pimprod2058310", "pimprod2058022"
    , "pimprod2058444", "pimprod2058937", "pimprod2059827", "pimprod2046731"
    , "pimprod2031369", "pimprod2059505", "pimprod2059504"  # a product, but don't have swatch_image_url
    , "pimprod2058875", "mkt77009341", "pimprod2030858", "pimprod2030331" # palette without swatch_image_url, or extremely small and unclear swatch_image_url
    , "pimprod2037999", "pimprod2043348", "pimprod2040913", "pimprod2042791", "pimprod2042794", "pimprod2042795", "pimprod2057307", "pimprod2058531" # palette without swatch_image_url, or extremely small and unclear swatch_image_ur
    , "pimprod2043657", "pimprod2050262", "pimprod2050001", "pimprod2052092", "pimprod2054173", "pimprod2054175", "pimprod2057441"# palette without swatch_image_url, or extremely small and unclear swatch_image_ur
    , "pimprod2042792" # pink blush and highlighter palette without swatch_image_url
    , "pimprod2042793" # orange blush and highlighter palette without swatch_image_url
    , "pimprod2050542"
]

## multi colors with makeup tools
mult_colors_tools = ["mkt77002543", "mkt77004018"]

# # set of trio of individual blush 
trio_combo = ["mkt77007685" ,"pimprod2034341", "pimprod2037930", "pimprod2045433", "pimprod2052960", "pimprod2047514"
              ,"pimprod2032372", "pimprod2038762", ]

# blush with a brush combo
blush_combo = ["mkt77004706", "pimprod2056457", "mkt77009832", "mkt77004705", "mkt77009894"]

small_swatch_image = ["pimprod2013408", "pimprod2045404", "pimprod2057533", "pimprod2057925", "pimprod2057923"] # extremely small and unclear swatch_image_url

### Stage 2

Uses `show_swatches()`, `remove_ids_from_list_v3()`, `check_swatch_image_url()`, `single_color_product()`, `remove_ids_from_list_v2()`, and `remove_ids_from_list()` to learn and reorganize the lists.

In [ ]:
palettes = {
    "no_swatch_image_url": ["pimprod2058531", "pimprod2057307", "pimprod2030858", "pimprod2040913", "pimprod2050262"
                            , "pimprod2058875", "pimprod2043657", "pimprod2037999", "pimprod2042795", "pimprod2042794" 
                            , "pimprod2050001", "pimprod2042791", "mkt77006098", "mkt77009341", "mkt77006104"
                            , "pimprod2042793" # palette in warm tone
                            , "pimprod2042792" # palette in cool tone
                            , "pimprod2020448"
                            ],
    "with_swatch_image_url": ["pimprod2054173", "pimprod2054175", "pimprod2043348", "pimprod2043348", "pimprod2052092"
                              ,"pimprod2030331", "pimprod2057441", "mkt77004019", "xlsImpprod14521197", "pimprod2059581"
                              ], 
}
# FINISH

In [ ]:
# # each swatch image has more than 1 shade of color
# # multi_color_swatch_image = multi_color_no_null
# # del multi_color_no_null # delete list 

# # print(len(multi_color_swatch_image)) # 26

multi_color_swatch_image = ['pimprod2051122',
 'pimprod2058301',
 'pimprod2046620',
 'pimprod2055456',
 'pimprod2056650',
 'pimprod2037175',
 'pimprod2056649',
 'pimprod2059602',
 'pimprod2050058',
 'pimprod2049565',
 'pimprod2052290',
 'pimprod2053334',
 'pimprod2042910',
 'pimprod2050040',
 'pimprod2053436',
 'pimprod2050643',
 'pimprod2050867',
 'pimprod2043903',
 'mkt77000411',
 'mkt77001487',
 'mkt77001819',
 'mkt77001820',
 'mkt77002358',
 'mkt77004845',
 'mkt77007311']

multi_color_swatch_image.append("mkt77008621") if "mkt77008621" not in multi_color_swatch_image else None
multi_color_swatch_image.append("pimprod2050338") if "pimprod2050338" not in multi_color_swatch_image else None 
multi_color_swatch_image.append("pimprod2057533") if "pimprod2057533" not in multi_color_swatch_image else None

multi_color_swatch_image.append("pimprod2047137") if "pimprod2047137" not in multi_color_swatch_image else None
multi_color_swatch_image.append("pimprod2047143") if "pimprod2047143" not in multi_color_swatch_image else None
multi_color_swatch_image.append("pimprod2047144") if "pimprod2047144" not in multi_color_swatch_image else None
multi_color_swatch_image.append("pimprod2050901") if "pimprod2050901" not in multi_color_swatch_image else None
multi_color_swatch_image.append("pimprod2043378") if "pimprod2043378" not in multi_color_swatch_image else None
multi_color_swatch_image.append("pimprod2044033") if "pimprod2044033" not in multi_color_swatch_image else None
multi_color_swatch_image.append("mkt77007645") if "mkt77007645" not in multi_color_swatch_image else None

print(len(multi_color_swatch_image)) # 35

# Removing existing element from other lists
set_blush = remove_ids_from_list_v3(set_blush, multi_color_swatch_image)
small_swatch_image = remove_ids_from_list_v3(small_swatch_image, multi_color_swatch_image)
remaining_product_ids = remove_ids_from_list_v3(remaining_product_ids, multi_color_swatch_image)

# FINISH 

In [503]:
print(len(multi_color_swatch_image))
print(multi_color_swatch_image)

35
['pimprod2051122', 'pimprod2058301', 'pimprod2046620', 'pimprod2055456', 'pimprod2056650', 'pimprod2037175', 'pimprod2056649', 'pimprod2059602', 'pimprod2050058', 'pimprod2049565', 'pimprod2052290', 'pimprod2053334', 'pimprod2042910', 'pimprod2050040', 'pimprod2053436', 'pimprod2050643', 'pimprod2050867', 'pimprod2043903', 'mkt77000411', 'mkt77001487', 'mkt77001819', 'mkt77001820', 'mkt77002358', 'mkt77004845', 'mkt77007311', 'mkt77008621', 'pimprod2050338', 'pimprod2057533', 'pimprod2047137', 'pimprod2047143', 'pimprod2047144', 'pimprod2050901', 'pimprod2043378', 'pimprod2044033', 'mkt77007645']


In [ ]:
single_color_no_swatch_image = ["mkt77005997", "mkt77009888", "pimprod2037136", "pimprod2042602", "pimprod2043762"
                                , "pimprod2048878", "pimprod2055993", "pimprod2059503", "xlsImpprod15771001", "pimprod2059505"
                                , "pimprod2059504", "pimprod2031369" , "pimprod2057200"
                                ]
set_blush = remove_ids_from_list_v3(set_blush, single_color_no_swatch_image)
remaining_product_ids = remove_ids_from_list_v3(remaining_product_ids, single_color_no_swatch_image)

#FINISH

In [459]:
single_color_swatch_image = [ "pimprod2056002"] # beginning

multi_color_swatch_image = remove_ids_from_list_v3(multi_color_swatch_image, single_color_swatch_image)
for i in small_swatch_image:
    single_color_swatch_image.append(i)
del small_swatch_image


# single_color_swatch_image = [
#             'pimprod2056002',
#             'pimprod2013408',
#             'pimprod2045404',
#             'pimprod2057925',
#             'pimprod2057923']

for i in remaining_product_ids:
    single_color_swatch_image.append(i)
print(len(single_color_swatch_image))
del remaining_product_ids

# latest
# single_color_swatch_image = ['pimprod2056002', 'pimprod2013408', 'pimprod2045404', 'pimprod2057925', 'pimprod2057923', 'VP10730'
#                              , 'VP12620', 'mkt77000420', 'mkt77000967', 'mkt77001349', 'mkt77001376', 'mkt77001962', 'mkt77002357'
#                              , 'mkt77002359', 'mkt77002360', 'mkt77002361', 'mkt77002432', 'mkt77004458', 'mkt77004460', 'mkt77005352'
#                              , 'mkt77005996', 'mkt77006982', 'mkt77007308', 'mkt77007498', 'mkt77007646', 'mkt77008314', 'mkt77008719'
#                              , 'mkt77008971', 'mkt77009340', 'mkt77009490', 'mkt77009882', 'mkt77009883', 'mkt77009884', 'pimprod2003079'
#                              , 'pimprod2004095', 'pimprod2005786', 'pimprod2007329', 'pimprod2008308', 'pimprod2009396', 'pimprod2012166'
#                              , 'pimprod2014270', 'pimprod2015177', 'pimprod2015889', 'pimprod2020788', 'pimprod2021060', 'pimprod2021340'
#                              , 'pimprod2024399', 'pimprod2024993', 'pimprod2025280', 'pimprod2027378', 'pimprod2028223', 'pimprod2028590'
#                              , 'pimprod2029892', 'pimprod2030849', 'pimprod2031376', 'pimprod2031864', 'pimprod2032200', 'pimprod2032207'
#                              , 'pimprod2032233', 'pimprod2032976', 'pimprod2033325', 'pimprod2033817', 'pimprod2034113', 'pimprod2034191'
#                              , 'pimprod2034497', 'pimprod2034815', 'pimprod2036986', 'pimprod2037375', 'pimprod2037387', 'pimprod2037598'
#                              , 'pimprod2037802', 'pimprod2038007', 'pimprod2038305', 'pimprod2038505', 'pimprod2038538', 'pimprod2038825'
#                              , 'pimprod2038948', 'pimprod2039276', 'pimprod2039299', 'pimprod2039629', 'pimprod2039778', 'pimprod2040099'
#                              , 'pimprod2040233', 'pimprod2040322', 'pimprod2040432', 'pimprod2040437', 'pimprod2040539', 'pimprod2040696'
#                              , 'pimprod2042625', 'pimprod2042810', 'pimprod2042919', 'pimprod2042954', 'pimprod2043349', 'pimprod2043369'
#                              , 'pimprod2043808', 'pimprod2043819', 'pimprod2043833', 'pimprod2043868', 'pimprod2043960', 'pimprod2044063'
#                              , 'pimprod2044097', 'pimprod2044285', 'pimprod2044741', 'pimprod2044820', 'pimprod2045159', 'pimprod2045180'
#                              , 'pimprod2045186', 'pimprod2045294', 'pimprod2045333', 'pimprod2045365', 'pimprod2045480', 'pimprod2045569'
#                              , 'pimprod2045574', 'pimprod2045615', 'pimprod2046142', 'pimprod2046149', 'pimprod2046212', 'pimprod2046323'
#                              , 'pimprod2046332', 'pimprod2046549', 'pimprod2046556', 'pimprod2046589', 'pimprod2046733', 'pimprod2047076'
#                              , 'pimprod2047199', 'pimprod2047239', 'pimprod2047580', 'pimprod2047872', 'pimprod2048094', 'pimprod2048380'
#                              , 'pimprod2048459', 'pimprod2049311', 'pimprod2049321', 'pimprod2049512', 'pimprod2049531', 'pimprod2049564'
#                              , 'pimprod2049596', 'pimprod2049677', 'pimprod2049768', 'pimprod2049930', 'pimprod2050045', 'pimprod2050543'
#                              , 'pimprod2050561', 'pimprod2050854', 'pimprod2050897', 'pimprod2051436', 'pimprod2051545', 'pimprod2052046'
#                              , 'pimprod2052296', 'pimprod2052324', 'pimprod2052350', 'pimprod2052391', 'pimprod2052464', 'pimprod2052620'
#                              , 'pimprod2052742', 'pimprod2052951', 'pimprod2053038', 'pimprod2053097', 'pimprod2053170', 'pimprod2053366'
#                              , 'pimprod2053369', 'pimprod2053417', 'pimprod2053450', 'pimprod2053552', 'pimprod2053617', 'pimprod2054162'
#                              , 'pimprod2054165', 'pimprod2054924', 'pimprod2055435', 'pimprod2055446', 'pimprod2055750', 'pimprod2055755'
#                              , 'pimprod2055865', 'pimprod2055880', 'pimprod2055883', 'pimprod2055989', 'pimprod2056106', 'pimprod2056116'
#                              , 'pimprod2056377', 'pimprod2056380', 'pimprod2056483', 'pimprod2056506', 'pimprod2056728', 'pimprod2056735'
#                              , 'pimprod2056792', 'pimprod2056852', 'pimprod2056913', 'pimprod2057244', 'pimprod2057251', 'pimprod2057260'
#                              , 'pimprod2057281', 'pimprod2057423', 'pimprod2057437', 'pimprod2057502', 'pimprod2057549', 'pimprod2057553'
#                              , 'pimprod2057554', 'pimprod2057728', 'pimprod2057785', 'pimprod2057953', 'pimprod2058427', 'pimprod2058473'
#                              , 'pimprod2058639', 'pimprod2058641', 'pimprod2058716', 'pimprod2058724', 'pimprod2058871', 'pimprod2058943'
#                              , 'pimprod2058946', 'pimprod2058961', 'pimprod2058964', 'pimprod2059137', 'pimprod2059224', 'pimprod2059619'
#                              , 'pimprod2059680', 'pimprod2059686', 'pimprod2060020', 'xlsImpprod10791937', 'xlsImpprod10791941'
#                              , 'xlsImpprod13361015', 'xlsImpprod13521181', 'xlsImpprod13741141', 'xlsImpprod15361021'
#                              , 'xlsImpprod15581009', 'xlsImpprod15581011', 'xlsImpprod15581029', 'xlsImpprod15711055'
#                              , 'xlsImpprod15921186', 'xlsImpprod16211171', 'xlsImpprod16281007', 'xlsImpprod16731045'
#                              , 'xlsImpprod18001081', 'xlsImpprod3650106', 'xlsImpprod820350']

#FINISH

234


In [ ]:
for i in mult_colors_tools:
    set_blush.append(i)
del mult_colors_tools

for i in trio_combo:
    set_blush.append(i)
del trio_combo

for i in blush_combo:
    set_blush.append(i)

# set_blush = [
#     'mkt77002598', 'mkt77008994', 'pimprod2018302', 'pimprod2037933', 'pimprod2053121'
#     , 'pimprod2057726', 'pimprod2058294', 'pimprod2058310', 'pimprod2058022', 'pimprod2058444'
#     , 'pimprod2058937', 'pimprod2059827', 'pimprod2046731', 'pimprod2050542', 'mkt77002543'
#     , 'mkt77004018', 'mkt77007685', 'pimprod2034341', 'pimprod2037930', 'pimprod2045433'
#     , 'pimprod2052960', 'pimprod2047514', 'pimprod2032372', 'pimprod2038762', 'mkt77004706'
#     , 'pimprod2056457', 'mkt77004705'
#     ]

# FINISH

In [480]:
print(len(set_blush))
set_blush

27


['mkt77002598',
 'mkt77008994',
 'pimprod2018302',
 'pimprod2037933',
 'pimprod2053121',
 'pimprod2057726',
 'pimprod2058294',
 'pimprod2058310',
 'pimprod2058022',
 'pimprod2058444',
 'pimprod2058937',
 'pimprod2059827',
 'pimprod2046731',
 'pimprod2050542',
 'mkt77002543',
 'mkt77004018',
 'mkt77007685',
 'pimprod2034341',
 'pimprod2037930',
 'pimprod2045433',
 'pimprod2052960',
 'pimprod2047514',
 'pimprod2032372',
 'pimprod2038762',
 'mkt77004706',
 'pimprod2056457',
 'mkt77004705']

## Check for all `product_id` are in a containers

In [499]:
# Step 1: Get all unique product_ids from variants_instock dataframe
unique_variant_ids = set(variants_instock['product_id'].unique())
print(f"Number of Unique product_id of variants_instock: {len(unique_variant_ids)}")

Number of Unique product_id of variants_instock: 335


In [500]:
# Step 2: Combine all product_ids from all lists into a single set
all_list_ids = set()

# Add palette product IDs
if 'no_swatch_image_url' in palettes:
    all_list_ids.update(palettes['no_swatch_image_url'])
if 'with_swatch_image_url' in palettes:
    all_list_ids.update(palettes['with_swatch_image_url'])

# Add other lists
all_list_ids.update(multi_color_swatch_image)
all_list_ids.update(single_color_no_swatch_image)
all_list_ids.update(single_color_swatch_image)
all_list_ids.update(set_blush)

print(f"Number of product_id in a container: {len(all_list_ids)}")

Number of product_id in a container: 336


In [501]:
# Step 3: Check if all variant product_ids are in the combined set
missing_ids = unique_variant_ids - all_list_ids

if missing_ids:
    print(f"Found {len(missing_ids)} product_ids in variants that are not in any list:")
    print(missing_ids)
else:
    print("All product_ids from variants are present in at least one list.")

# Optional: Get the variants that are missing
missing_variants = variants_instock[variants_instock['product_id'].isin(missing_ids)]

All product_ids from variants are present in at least one list.


## Create DataFrames and Export Files

In [504]:
import os

In [510]:
# Optional: Create an output directory
output_dir = "../data/processed_data/test/blushes"
os.makedirs(output_dir, exist_ok=True)

In [511]:
# Create dataframes and export
categories = {
    'palettes_no_swatch_image': palettes['no_swatch_image_url'],
    'palettes_with_swatch_image': palettes['with_swatch_image_url'],
    'multi_color_swatch_image': multi_color_swatch_image,
    'single_color_no_swatch_image': single_color_no_swatch_image,
    'single_color_swatch_image': single_color_swatch_image,
    'set_blush': set_blush
}

In [512]:
# Summary dictionary to track results
summary = {}

for category_name, product_ids in categories.items():
    # Convert product_ids to match the type in variants_instock['product_id']
    # This ensures proper matching regardless of string/int differences
    product_ids = [str(pid) for pid in product_ids]
    
    # Filter variants_instock where product_id matches (converting to string for comparison)
    df = variants_instock[variants_instock['product_id'].astype(str).isin(product_ids)]
    
    # Export to CSV
    filepath = os.path.join(output_dir, f"{category_name}.csv")
    df.to_csv(filepath, index=False)
    
    # Store summary
    summary[category_name] = {
        'rows': len(df),
        'unique_products': df['product_id'].nunique(),
        'file': filepath
    }
    
    print(f"✓ {category_name}: {len(df)} rows, {df['product_id'].nunique()} unique products → {filepath}")

✓ palettes_no_swatch_image: 17 rows, 17 unique products → ../data/processed_data/test/blushes\palettes_no_swatch_image.csv
✓ palettes_with_swatch_image: 35 rows, 9 unique products → ../data/processed_data/test/blushes\palettes_with_swatch_image.csv
✓ multi_color_swatch_image: 168 rows, 35 unique products → ../data/processed_data/test/blushes\multi_color_swatch_image.csv
✓ single_color_no_swatch_image: 13 rows, 13 unique products → ../data/processed_data/test/blushes\single_color_no_swatch_image.csv
✓ single_color_swatch_image: 1365 rows, 234 unique products → ../data/processed_data/test/blushes\single_color_swatch_image.csv
✓ set_blush: 65 rows, 27 unique products → ../data/processed_data/test/blushes\set_blush.csv


In [513]:
# Print summary
print("\n" + "="*50)
print("EXPORT SUMMARY")
print("="*50)
total_rows = sum(s['rows'] for s in summary.values())
print(f"Total categories: {len(summary)}")
print(f"Total rows exported: {total_rows}")
print(f"Files saved in: {os.path.abspath(output_dir)}")

# Optional: Create a combined summary CSV
summary_df = pd.DataFrame([
    {'category': name, **stats} 
    for name, stats in summary.items()
])
summary_df.to_csv(os.path.join(output_dir, '_export_summary.csv'), index=False)
print("Summary file: _export_summary.csv")


EXPORT SUMMARY
Total categories: 6
Total rows exported: 1663
Files saved in: c:\Users\Thao\Documents\Ulta Analysis\data\processed_data\test\blushes
Summary file: _export_summary.csv


## Missing Size

In [520]:
# Find unique product_ids where size_text is missing (NaN or None)
missing_size_text = variants_instock[
    variants_instock['size_text'].isna() | 
    (variants_instock['size_text'] == '') | 
    (variants_instock['size_text'].isnull())
]['product_id'].unique()

print(f"Found {len(missing_size_text)} unique product_ids with missing size_text:")
print(missing_size_text)

# If you want to see them in a more readable format
if len(missing_size_text) > 0:
    print("\nProduct IDs with missing size_text:")
    for pid in missing_size_text:
        print(f"  - {pid}")

Found 80 unique product_ids with missing size_text:
<StringArray>
['pimprod2025280', 'pimprod2054173', 'pimprod2056002', 'pimprod2056457',
 'pimprod2058531', 'pimprod2057554', 'pimprod2050897', 'pimprod2043833',
 'pimprod2057553', 'pimprod2058937', 'pimprod2020448', 'pimprod2054175',
 'pimprod2037387', 'pimprod2050542', 'pimprod2047076', 'pimprod2056913',
 'pimprod2055435', 'pimprod2059827', 'pimprod2059602', 'pimprod2052092',
 'pimprod2046549', 'pimprod2058724', 'pimprod2050001', 'pimprod2053617',
 'pimprod2048094', 'pimprod2045433', 'pimprod2042810', 'pimprod2052290',
 'pimprod2059680', 'pimprod2053097', 'pimprod2058964', 'pimprod2054165',
 'pimprod2058716', 'pimprod2052464', 'pimprod2033817', 'pimprod2058022',
 'pimprod2032372', 'pimprod2057307', 'pimprod2030858', 'pimprod2052960',
 'pimprod2058444', 'pimprod2037933', 'pimprod2057441', 'pimprod2050643',
 'pimprod2043762', 'pimprod2043657', 'pimprod2037930', 'pimprod2059686',
 'pimprod2058310', 'pimprod2053121', 'pimprod2050045', 'pi

In [521]:
missing_size_text

<StringArray>
['pimprod2025280', 'pimprod2054173', 'pimprod2056002', 'pimprod2056457',
 'pimprod2058531', 'pimprod2057554', 'pimprod2050897', 'pimprod2043833',
 'pimprod2057553', 'pimprod2058937', 'pimprod2020448', 'pimprod2054175',
 'pimprod2037387', 'pimprod2050542', 'pimprod2047076', 'pimprod2056913',
 'pimprod2055435', 'pimprod2059827', 'pimprod2059602', 'pimprod2052092',
 'pimprod2046549', 'pimprod2058724', 'pimprod2050001', 'pimprod2053617',
 'pimprod2048094', 'pimprod2045433', 'pimprod2042810', 'pimprod2052290',
 'pimprod2059680', 'pimprod2053097', 'pimprod2058964', 'pimprod2054165',
 'pimprod2058716', 'pimprod2052464', 'pimprod2033817', 'pimprod2058022',
 'pimprod2032372', 'pimprod2057307', 'pimprod2030858', 'pimprod2052960',
 'pimprod2058444', 'pimprod2037933', 'pimprod2057441', 'pimprod2050643',
 'pimprod2043762', 'pimprod2043657', 'pimprod2037930', 'pimprod2059686',
 'pimprod2058310', 'pimprod2053121', 'pimprod2050045', 'pimprod2014270',
 'pimprod2042795', 'pimprod2046731', 

In [524]:
# Show all rows where size_text is missing
missing_size_text_df = variants_instock[
    variants_instock['size_text'].isna() | 
    (variants_instock['size_text'] == '') | 
    (variants_instock['size_text'].isnull())
]

print(f"Total rows with missing size_text: {len(missing_size_text_df)}")
print(f"Unique product_ids affected: {missing_size_text_df['product_id'].nunique()}")

# Display the first few rows
print("\nSample rows with missing size_text:")
print(missing_size_text_df[['product_id', 'size_text']].head(10))

# Group by product_id to see how many missing entries per product
product_missing_counts = missing_size_text_df.groupby('product_id').size().reset_index(name='missing_count')
print("\nMissing size_text count per product_id:")
product_missing_counts

Total rows with missing size_text: 257
Unique product_ids affected: 80

Sample rows with missing size_text:
         product_id size_text
44   pimprod2025280       NaN
45   pimprod2025280       NaN
46   pimprod2025280       NaN
272  pimprod2054173       NaN
273  pimprod2054173       NaN
274  pimprod2054173       NaN
275  pimprod2054173       NaN
276  pimprod2054173       NaN
277  pimprod2056002       NaN
278  pimprod2056002       NaN

Missing size_text count per product_id:


,product_id,missing_count
0,mkt77002357,8
1,mkt77002358,3
2,mkt77002359,6
3,mkt77002360,6
4,mkt77002361,5
...,...,...
75,pimprod2058964,6
76,pimprod2059602,4
77,pimprod2059680,6
78,pimprod2059686,6


In [525]:
# Simple one-liner to get unique product_ids
unique_missing = variants_instock[variants_instock['size_text'].isna() | (variants_instock['size_text'] == '')]['product_id'].unique()
print(f"Unique product_ids with missing size_text: {len(unique_missing)}")
print(unique_missing)

Unique product_ids with missing size_text: 80
<StringArray>
['pimprod2025280', 'pimprod2054173', 'pimprod2056002', 'pimprod2056457',
 'pimprod2058531', 'pimprod2057554', 'pimprod2050897', 'pimprod2043833',
 'pimprod2057553', 'pimprod2058937', 'pimprod2020448', 'pimprod2054175',
 'pimprod2037387', 'pimprod2050542', 'pimprod2047076', 'pimprod2056913',
 'pimprod2055435', 'pimprod2059827', 'pimprod2059602', 'pimprod2052092',
 'pimprod2046549', 'pimprod2058724', 'pimprod2050001', 'pimprod2053617',
 'pimprod2048094', 'pimprod2045433', 'pimprod2042810', 'pimprod2052290',
 'pimprod2059680', 'pimprod2053097', 'pimprod2058964', 'pimprod2054165',
 'pimprod2058716', 'pimprod2052464', 'pimprod2033817', 'pimprod2058022',
 'pimprod2032372', 'pimprod2057307', 'pimprod2030858', 'pimprod2052960',
 'pimprod2058444', 'pimprod2037933', 'pimprod2057441', 'pimprod2050643',
 'pimprod2043762', 'pimprod2043657', 'pimprod2037930', 'pimprod2059686',
 'pimprod2058310', 'pimprod2053121', 'pimprod2050045', 'pimprod2

In [532]:
# Get unique product_ids with missing size_text
missing_size_ids = variants_instock[
    variants_instock['size_text'].isna() | 
    (variants_instock['size_text'] == '') | 
    (variants_instock['size_text'].isnull())
]['product_id'].unique()

# Convert to set for easier operations
missing_size_set = set(str(pid) for pid in missing_size_ids)

# Define your categories
categories = {
    'palettes_no_swatch_image': palettes['no_swatch_image_url'],
    'palettes_with_swatch_image': palettes['with_swatch_image_url'],
    'multi_color_swatch_image': multi_color_swatch_image,
    'single_color_no_swatch_image': single_color_no_swatch_image,
    'single_color_swatch_image': single_color_swatch_image,
    'set_blush': set_blush
}

# Check each category
print("Missing size_text product_ids in each category:")
print("="*60)

total_missing_found = 0

for category_name, product_ids in categories.items():
    # Convert category product_ids to strings for comparison
    category_ids = set(str(pid) for pid in product_ids)
    
    # Find intersection - missing size_text IDs that are in this category
    missing_in_category = missing_size_set.intersection(category_ids)
    
    count = len(missing_in_category)
    total_missing_found += count
    
    print(f"\n{category_name}:")
    print(f"  Total products in category: {len(category_ids)}")
    print(f"  Products with missing size_text: {count}")
    
    if count > 0:
        print(f"  Product IDs: {sorted(missing_in_category)}")

print("\n" + "="*60)
print(f"Total unique missing size_text products: {len(missing_size_set)}")
print(f"Total missing products found in categories: {total_missing_found}")

# Check if any missing products are NOT in any category
all_category_ids = set()
for product_ids in categories.values():
    all_category_ids.update(str(pid) for pid in product_ids)

unaccounted_missing = missing_size_set - all_category_ids
if unaccounted_missing:
    print(f"\n⚠️  {len(unaccounted_missing)} missing size_text products NOT in any category:")
    print(sorted(unaccounted_missing))

Missing size_text product_ids in each category:

palettes_no_swatch_image:
  Total products in category: 18
  Products with missing size_text: 8
  Product IDs: ['mkt77009341', 'pimprod2020448', 'pimprod2030858', 'pimprod2042795', 'pimprod2043657', 'pimprod2050001', 'pimprod2057307', 'pimprod2058531']

palettes_with_swatch_image:
  Total products in category: 9
  Products with missing size_text: 4
  Product IDs: ['pimprod2052092', 'pimprod2054173', 'pimprod2054175', 'pimprod2057441']

multi_color_swatch_image:
  Total products in category: 35
  Products with missing size_text: 5
  Product IDs: ['mkt77002358', 'pimprod2044033', 'pimprod2050643', 'pimprod2052290', 'pimprod2059602']

single_color_no_swatch_image:
  Total products in category: 13
  Products with missing size_text: 4
  Product IDs: ['mkt77005997', 'mkt77009888', 'pimprod2043762', 'pimprod2055993']

single_color_swatch_image:
  Total products in category: 234
  Products with missing size_text: 38
  Product IDs: ['mkt77002357'